In [1]:
import os
import sys
import importlib
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Ensure project root is importable
_project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

# ──── CONFIG ────
TICKER = "MSFT"
MODEL_DIR = f"output/{TICKER.lower()}"

print(f"Ticker: {TICKER} | Model dir: {MODEL_DIR}")

# Load the WaveNet Lambert GAN API
from generator.WAVENET_LAMBERT_GAN.models.API import GeneratorAPI

api = GeneratorAPI(model_dir=MODEL_DIR)
print(f"Config: {api.config}")

Ticker: MSFT | Model dir: output/msft


I0000 00:00:1776213430.654320 1780847 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


FileNotFoundError: Model directory not found: output/msft

In [ ]:
# ──── Generate synthetic data in different spaces ────

# 1. Lambert-scaled space (generator's native output)
synth_scaled = api.generate(n_samples=200)
print(f"Scaled (Lambert) — shape: {synth_scaled.shape}")
print(f"  range: [{synth_scaled.min():.4f}, {synth_scaled.max():.4f}]")
print(f"  mean:  {synth_scaled.mean():.4f}, std: {synth_scaled.std():.4f}")

# 2. Log-return space (inverse Lambert)
synth_lr = api.generate_log_returns(n_samples=200)
print(f"\nLog returns — shape: {synth_lr.shape}")
print(f"  range: [{synth_lr.min():.6f}, {synth_lr.max():.6f}]")
print(f"  mean:  {synth_lr.mean():.6f}, std: {synth_lr.std():.6f}")

# Compare with real log returns from training data
real_lr = api.log_returns
print(f"\nReal log returns — {len(real_lr)} observations")
print(f"  range: [{real_lr.min():.6f}, {real_lr.max():.6f}]")
print(f"  mean:  {real_lr.mean():.6f}, std: {real_lr.std():.6f}")

In [ ]:
# ──── Generate synthetic price paths ────
import matplotlib.pyplot as plt

prices = api.generate_prices(n_samples=20, initial_price=api.close_prices[-1])
print(f"Price paths — shape: {prices.shape}")
print(f"  Initial price: ${api.close_prices[-1]:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot price paths
for i in range(min(20, len(prices))):
    axes[0].plot(prices[i], alpha=0.5)
axes[0].set_title(f'{TICKER} — Synthetic Price Paths')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

# Plot log-return distributions: real vs synthetic
synth_lr_flat = api.generate_log_returns(n_samples=500).flatten()
axes[1].hist(real_lr, bins=100, alpha=0.5, density=True, label='Real', color='tab:blue')
axes[1].hist(synth_lr_flat, bins=100, alpha=0.5, density=True, label='Synthetic', color='tab:orange')
axes[1].set_title(f'{TICKER} — Log-Return Distribution')
axes[1].set_xlabel('Log Return')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

['CORN' 'CYB' 'DBB' 'DBC' 'FXA' 'FXB' 'FXC' 'FXE' 'FXY' 'GLD' 'IWM' 'NIB'
 'QQQ' 'SLV' 'SPY' 'UGA' 'UNG' 'USO']
ticker_index:  0
real_data:  (1, 120, 90)
target_macro:  (1, 120, 46)
processed_pv_feature all shape:  (1, 120, 90)
processed_pv_feature all shape:  (120, 90)
processed_pv_feature shape:  (120, 5)
open all shape:  (2397, 18)
close all shape:  (2397, 18, 120)
close shape:  ()
close 12.40999984741211
open 12.210000038146973
df_features shape:  (120, 5)
original_close:  12.40999984741211
original_open:  12.210000038146973
smoothed_close shape:  (120,)
smoothed_open shape:  (120,)
smoothed_high shape:  (120,)
smoothed_low shape:  (120,)
open:  0      1.221000e+01
1      1.383760e+00
2      9.394402e+00
3      2.121274e+01
4     -4.958565e+00
           ...     
115   -1.174590e-16
116   -1.755200e-16
117   -4.982822e-16
118   -5.558472e-16
119   -1.336656e-15
Length: 120, dtype: float64
close:  0      1.241000e+01
1      1.441336e+01
2      9.293713e+00
3      1.930116e+01
4     

/data3/hcxia/Adahist2/generator/GRT_GAN/models/API.py:275: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["cord_{}".format(w)] = df1.rolling(w).corr(pairwise = df2.rolling(w))
/data3/hcxia/Adahist2/generator/GRT_GAN/models/API.py:277: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['abs_ret1'] = np.abs(df['ret1'])
/data3/hcxia/Adahist2/generator/GRT_GAN/models/API.py:278: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider 

In [ ]:
# ──── Statistical comparison (summary stats, autocorrelation) ────
from generator.WAVENET_LAMBERT_GAN.models.train import preprocess_lambert, make_sequences

# Build real sequences in Lambert space using same preprocessing
scaled, _, _, _ = preprocess_lambert(api.log_returns)
real_sequences = make_sequences(scaled, api.seq_len)
fake_sequences = api.generate(n_samples=len(real_sequences))

print(f"Real sequences:  {real_sequences.shape}")
print(f"Fake sequences:  {fake_sequences.shape}")

# Marginal stats comparison
from scipy import stats
real_flat = real_sequences.flatten()
fake_flat = fake_sequences.flatten()

print(f"\n{'Metric':<12} {'Real':>10} {'Synthetic':>10} {'|Diff|':>10}")
print("-" * 44)
for name, fn in [('Mean', np.mean), ('Std', np.std),
                 ('Skew', lambda x: float(stats.skew(x))),
                 ('Kurtosis', lambda x: float(stats.kurtosis(x)))]:
    r, f = fn(real_flat), fn(fake_flat)
    print(f"{name:<12} {r:10.4f} {f:10.4f} {abs(r-f):10.4f}")

[[Timestamp('2011-01-03 00:00:00') Timestamp('2011-01-03 00:00:00')
  Timestamp('2011-01-03 00:00:00') ... Timestamp('2011-01-03 00:00:00')
  Timestamp('2011-01-03 00:00:00') Timestamp('2011-01-03 00:00:00')]
 [Timestamp('2011-01-04 00:00:00') Timestamp('2011-01-04 00:00:00')
  Timestamp('2011-01-04 00:00:00') ... Timestamp('2011-01-04 00:00:00')
  Timestamp('2011-01-04 00:00:00') Timestamp('2011-01-04 00:00:00')]
 [Timestamp('2011-01-05 00:00:00') Timestamp('2011-01-05 00:00:00')
  Timestamp('2011-01-05 00:00:00') ... Timestamp('2011-01-05 00:00:00')
  Timestamp('2011-01-05 00:00:00') Timestamp('2011-01-05 00:00:00')]
 ...
 [Timestamp('2020-07-09 00:00:00') Timestamp('2020-07-09 00:00:00')
  Timestamp('2020-07-09 00:00:00') ... Timestamp('2020-07-09 00:00:00')
  Timestamp('2020-07-09 00:00:00') Timestamp('2020-07-09 00:00:00')]
 [Timestamp('2020-07-10 00:00:00') Timestamp('2020-07-10 00:00:00')
  Timestamp('2020-07-10 00:00:00') ... Timestamp('2020-07-10 00:00:00')
  Timestamp('2020-0